# STEP 4 - PERSISTENT VECTOR STORAGE

In [ ]:
import sys
from pathlib import Path
PROJECT_ROOT = Path(".." ).resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

## helper method to reload specific file
import importlib
import config
importlib.reload(config)

In [ ]:
from pathlib import Path
import numpy as np
import gradio as gr
from openai import OpenAI
from config import OPENAI_API_KEY, MODEL_NAME, EMBEDDING_MODEL

# 1. Initialize client & resolve paths
client = OpenAI(api_key=OPENAI_API_KEY)

# 2. Load document & split into paragraphs
document_path = PROJECT_ROOT / "data" / "profile.txt"
document_text = document_path.read_text(encoding="utf-8")
paragraphs = [p.strip() for p in document_text.split("\n\n") if p.strip()]
documents = [{"id": i, "text": p} for i, p in enumerate(paragraphs)]

# 3. Compute embeddings for all document paragraphs
embedded_documents = []
for doc in documents:
    response = client.embeddings.create(
        model=EMBEDDING_MODEL,
        input=doc["text"]
    )
    embedded_documents.append({
        "id": doc["id"],
        "text": doc["text"],
        "embedding": response.data[0].embedding
    })

# 4. Helper functions for Cosine Similarity & Retrieval
def cosine_similarity(a, b):
    a = np.array(a)
    b = np.array(b)
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

def retrieve(query_vector, docs, top_k=3):
    scores = []
    for doc in docs:
        score = cosine_similarity(query_vector, doc["embedding"])
        scores.append({"id": doc["id"], "score": score, "text": doc["text"]})
    scores.sort(key=lambda x: x["score"], reverse=True)
    return scores[:top_k]

# 5. RAG Chatbot function for Gradio ChatInterface
def rag_chatbot(message, history):
    # Embed the user query
    response = client.embeddings.create(
        model=EMBEDDING_MODEL,
        input=message
    )
    query_vector = response.data[0].embedding
    
    # Retrieve top 3 relevant paragraphs
    top_docs = retrieve(query_vector, embedded_documents, top_k=3)
    retrieved_context = "\n\n".join([d["text"] for d in top_docs])
    
    # Build RAG system prompt
    system_prompt = f"""You answer questions using ONLY the retrieved context below.

If the answer is not present in the context, say:
'I could not find that information in the document.'

Retrieved Context:

{retrieved_context}"""

    messages = [{"role": "system", "content": system_prompt}]
    for user_msg, bot_msg in history:
        messages.append({"role": "user", "content": user_msg})
        messages.append({"role": "assistant", "content": bot_msg})
    messages.append({"role": "user", "content": message})
    
    chat_response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=messages
    )
    return chat_response.choices[0].message.content

# 6. Launch Gradio ChatInterface
demo = gr.ChatInterface(
    fn=rag_chatbot,
    title="Step 3: RAG Chatbot (Brute-Force Embedding Retrieval)",
    description="Ask questions about John Doe's profile. Uses vector embeddings & cosine similarity for top-k paragraph retrieval."
)

# demo.launch(prevent_thread_lock=True)

In [ ]:
demo.close()

In [ ]:
import chromadb

print(chromadb.__version__)

In [ ]:
client_chroma = chromadb.HttpClient(
    host="localhost",
    port=8000,
)

print("Connected!")

In [ ]:
collection= client_chroma.get_or_create_collection(
    name="john_doe_profile"
)

print(collection)

In [ ]:
collection.count()

In [ ]:
# DELETING THE COLLECTION FOR TESTING AND DEV PHASE
client.delete_collection("john_doe_profile")

collection = client_chroma.get_or_create_collection(
    name="john_doe_profile"
)

In [ ]:
collection.add(
    ids=[str(doc["id"]) for doc in embedded_documents],
    documents=[doc["text"] for doc in embedded_documents],
    embeddings=[doc["embedding"] for doc in embedded_documents],
)

In [ ]:
results = collection.get(limit=1)

results

In [ ]:
results = collection.get(
    ids=["0"],
    include=["documents", "embeddings", "metadatas"]
)

results

In [ ]:
collection.peek(limit=3)

In [ ]:
# TO AVOID ADDING THE SAME ID'S AGAIN IN THE DB
# upsert means:

# If the ID doesn't exist → insert it.
# If the ID already exists → update it.

collection.upsert(
    ids=[str(doc["id"]) for doc in embedded_documents],
    documents=[doc["text"] for doc in embedded_documents],
    embeddings=[doc["embedding"] for doc in embedded_documents],
)

In [ ]:
collection.count()

In [ ]:
query = "Where did John Doe study?"

response = client.embeddings.create(
    model=EMBEDDING_MODEL,
    input=query
)

query_embedding = response.data[0].embedding

print(len(query_embedding))

In [ ]:
results = collection.query(
    query_embeddings=[query_embedding],
    n_results=3
)

In [ ]:
from pprint import pprint

pprint(results)

In [ ]:
ids = results["ids"][0]
documents = results["documents"][0]
distances = results["distances"][0]
metadatas = results["metadatas"][0]

In [ ]:
similarities = [
    (1 - distance) * 100
    for distance in distances
]

In [ ]:
for i in range(len(ids)):
    print("=" * 60)
    print(f"ID: {ids[i]}")
    print(f"Similarity: {similarities[i]:.2f}%")
    print(f"Distance: {distances[i]:.4f}")
    print("Metadata:", metadatas[i])
    print()
    print(documents[i])

## CHATBOT USING VECTOR DB - CHROMA 
instead of checking every query from the python embedded list

In [ ]:
import sys
from pathlib import Path
PROJECT_ROOT = Path(".." ).resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

## helper method to reload specific file
import importlib
import config
importlib.reload(config)

In [ ]:
from pathlib import Path
import numpy as np
import gradio as gr
from openai import OpenAI
from config import OPENAI_API_KEY, MODEL_NAME, EMBEDDING_MODEL
import chromadb

# 1. Initialize client & resolve paths
client = OpenAI(api_key=OPENAI_API_KEY)
client_chroma= chromadb.HttpClient(
    host="localhost",
    port=8000,
)

In [ ]:
# 2. Load document & split into paragraphs
document_path = PROJECT_ROOT / "data" / "profile.txt"
document_text = document_path.read_text(encoding="utf-8")
paragraphs = [p.strip() for p in document_text.split("\n\n") if p.strip()]
documents = [{"id": i, "text": p} for i, p in enumerate(paragraphs)]

In [ ]:
# 3. Compute embeddings for all document paragraphs
embedded_documents = []
for doc in documents:
    response = client.embeddings.create(
        model=EMBEDDING_MODEL,
        input=doc["text"]
    )
    embedded_documents.append({
        "id": doc["id"],
        "text": doc["text"],
        "embedding": response.data[0].embedding
    })

In [ ]:
# 4. Create/get collection from DB
collection= client_chroma.get_or_create_collection(
    name="john_doe_profile"
)

# 5. Adding the embedded data in the DB
collection.add(
    ids=[str(doc["id"]) for doc in embedded_documents],
    documents=[doc["text"] for doc in embedded_documents],
    embeddings=[doc["embedding"] for doc in embedded_documents],
)

In [ ]:
# 6. RAG chatbot function for Gradio

def rag_chatbot(message,history):
    response = client.embeddings.create(
            model=EMBEDDING_MODEL,
            input=message
        )
    query_vector = response.data[0].embedding

    # Quering Vector DB
    result = collection.query(
        query_embeddings=[query_vector],
        n_results=3
    )

    # Retrieving the data we got from the response 
    retrieved_chunks=result["documents"][0]
    retrieved_context="\n\n".join(retrieved_chunks)
    

    # Build RAG system prompt
    system_prompt = f"""You answer questions using ONLY the retrieved context below.

        If the answer is not present in the context, say:
        'I could not find that information in the document.'
        Retrieved Context:
        {retrieved_context}
"""

    messages = [{"role": "system", "content": system_prompt}]
    for user_msg, bot_msg in history:
        messages.append({"role": "user", "content": user_msg})
        messages.append({"role": "assistant", "content": bot_msg})
    messages.append({"role": "user", "content": message})
    
    chat_response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=messages
    )
    return chat_response.choices[0].message.content

In [ ]:
# 7. Launch Gradio ChatInterface
demo = gr.ChatInterface(
    fn=rag_chatbot,
    title="Step 3: RAG Chatbot (Chroma - VECTOR DB)",
    description="Ask questions about John Doe's profile."
)

In [ ]:
demo.launch(prevent_thread_lock=True)

In [ ]:
demo.close()

## 7. Display Retrieved Results with Expandable Debug Interface

In [68]:
import time

def query_with_debug(query_text, top_k=3):
    start_time = time.perf_counter()
    
    # 1. Embed query
    response = client.embeddings.create(
        model=EMBEDDING_MODEL,
        input=query_text
    )
    query_vector = response.data[0].embedding
    
    # 2. Query Vector DB collection
    results = collection.query(
        query_embeddings=[query_vector],
        n_results=top_k,
        include=["documents", "distances", "metadatas"]
    )
    
    latency_ms = (time.perf_counter() - start_time) * 1000
    
    ids = results["ids"][0]
    documents = results["documents"][0]
    distances = results["distances"][0]
    metadatas = results["metadatas"][0] if results.get("metadatas") else [{}] * len(ids)
    
    # Print formatted debug summary
    print(f"🔍 Query: '{query_text}'")
    print(f"⏱️ Query Latency: {latency_ms:.2f} ms")
    print("=" * 85)
    print(f"{'Rank':<5} | {'Chunk ID':<10} | {'Source File':<15} | {'Distance':<10} | {'Similarity':<12} | {'Retrieved Text Snippet':<30}")
    print("-" * 85)
    
    for rank, (chunk_id, doc_text, dist, meta) in enumerate(zip(ids, documents, distances, metadatas), start=1):
        similarity_pct = max(0.0, (1 - dist) * 100)
        source_file = meta.get("source", "profile.txt") if (meta and isinstance(meta, dict)) else "profile.txt"
        short_text = doc_text.replace("\n", " ")[:35] + ("..." if len(doc_text) > 35 else "")
        print(f"{rank:<5} | {str(chunk_id):<10} | {source_file:<15} | {dist:<10.4f} | {f'{similarity_pct:.2f}%':<12} | {short_text}")
    print("=" * 85 + "\n")
    
    return results

# Example Direct Query Calls:
results1 = query_with_debug("Where did John Doe study?")
results2 = query_with_debug("What programming languages does John know?")


🔍 Query: 'Where did John Doe study?'
⏱️ Query Latency: 1934.71 ms
Rank  | Chunk ID   | Source File     | Distance   | Similarity   | Retrieved Text Snippet        
-------------------------------------------------------------------------------------
1     | 1          | profile.txt     | 0.7882     | 21.18%       | John Alexander Doe was born on Marc...
2     | 2          | profile.txt     | 0.8563     | 14.37%       | John was the eldest of three childr...
3     | 0          | profile.txt     | 0.9106     | 8.94%        | Fictional Biography of John Doe Joh...

🔍 Query: 'What programming languages does John know?'
⏱️ Query Latency: 579.00 ms
Rank  | Chunk ID   | Source File     | Distance   | Similarity   | Retrieved Text Snippet        
-------------------------------------------------------------------------------------
1     | 4          | profile.txt     | 0.7828     | 21.72%       | By the time he entered high school,...
2     | 9          | profile.txt     | 0.7829     | 21.71% 

## 8. Restart Application & Query Existing Collection (Skip Embedding Generation)

In [69]:
# 1. Simulate restarting application: reconnect to existing ChromaDB client
client_chroma_reconnect = chromadb.HttpClient(host="localhost", port=8000)

# 2. Get existing collection without calling collection.add() or generating document embeddings again
existing_collection = client_chroma_reconnect.get_collection(name="john_doe_profile")

# 3. Confirm that documents and embeddings remain stored
doc_count = existing_collection.count()
print(f"✅ Reconnected successfully!")
print(f"Total stored documents in 'john_doe_profile': {doc_count}")

# 4. Query existing collection directly
test_query = "What is John Doe's educational background?"
query_resp = client.embeddings.create(model=EMBEDDING_MODEL, input=test_query)
query_vector = query_resp.data[0].embedding

res = existing_collection.query(
    query_embeddings=[query_vector],
    n_results=2
)

print("\n--- Query Results from Persistent Store ---")
for doc_id, text in zip(res["ids"][0], res["documents"][0]):
    print(f"ID: {doc_id} | Snippet: {text[:80]}...")

✅ Reconnected successfully!
Total stored documents in 'john_doe_profile': 22

--- Query Results from Persistent Store ---
ID: 1 | Snippet: John Alexander Doe was born on March 18, 1987, in the quiet town of Brookfield, ...
ID: 2 | Snippet: John was the eldest of three children born to Michael and Sarah Doe. His father ...


## 9. Test Metadata Filters

In [70]:
# 1. Create a metadata-enabled collection for testing filtering
metadata_collection = client_chroma.get_or_create_collection(name="company_policy_test")

# Clear old entries if re-running
try:
    existing_ids = metadata_collection.get()["ids"]
    if existing_ids:
        metadata_collection.delete(ids=existing_ids)
except Exception:
    pass

# 2. Define multiple reference documents with structured metadata (document_type and department)
policy_docs = [
    {
        "id": "policy_finance_1",
        "text": "Company Travel Policy: International travel requests above $2000 require CFO and Finance department approval.",
        "metadata": {"document_type": "company_policy", "department": "finance", "source": "travel_policy.txt"}
    },
    {
        "id": "policy_finance_2",
        "text": "Expense Reimbursement Policy: All expense claims must be filed within 14 days with itemized receipts to Finance.",
        "metadata": {"document_type": "company_policy", "department": "finance", "source": "expense_policy.txt"}
    },
    {
        "id": "policy_hr_1",
        "text": "Remote Work Guidelines: Employees may work remotely up to 2 days per week after HR clearance.",
        "metadata": {"document_type": "company_policy", "department": "hr", "source": "remote_policy.txt"}
    },
    {
        "id": "guide_eng_1",
        "text": "Engineering Infrastructure Architecture: All microservices must be deployed using Docker and Kubernetes.",
        "metadata": {"document_type": "tech_guide", "department": "engineering", "source": "tech_architecture.txt"}
    }
]

# 3. Embed documents and store in Chroma with metadata
doc_texts = [d["text"] for d in policy_docs]
embeddings_res = client.embeddings.create(model=EMBEDDING_MODEL, input=doc_texts)

metadata_collection.add(
    ids=[d["id"] for d in policy_docs],
    documents=doc_texts,
    embeddings=[item.embedding for item in embeddings_res.data],
    metadatas=[d["metadata"] for d in policy_docs]
)

print(f"✅ Indexed {metadata_collection.count()} documents with metadata tags.\n")

# 4. Helper function to test queries with/without metadata filters
def run_filtered_query(query_text, where_filter=None):
    q_emb = client.embeddings.create(model=EMBEDDING_MODEL, input=query_text).data[0].embedding
    
    query_args = {"query_embeddings": [q_emb], "n_results": 3}
    if where_filter:
        query_args["where"] = where_filter
        
    res = metadata_collection.query(**query_args)
    
    filter_label = str(where_filter) if where_filter else "None (Unfiltered Search)"
    print(f"🔍 Query: '{query_text}' | Filter: {filter_label}")
    for rank, (doc_id, text, meta) in enumerate(zip(res["ids"][0], res["documents"][0], res["metadatas"][0]), start=1):
        print(f"   Rank {rank} | ID: {doc_id} | Dept: {meta.get('department')} | Doc Type: {meta.get('document_type')}")
        print(f"          Snippet: {text[:75]}...")
    print("=" * 75)

# Test 1: Run query WITHOUT filters
run_filtered_query("What are the policy guidelines and budget rules?", where_filter=None)

# Test 2: Filtered by document_type = 'company_policy'
run_filtered_query("What are the policy guidelines and budget rules?", where_filter={"document_type": "company_policy"})

# Test 3: Filtered by department = 'finance'
run_filtered_query("What are the policy guidelines and budget rules?", where_filter={"department": "finance"})

# Test 4: Filtered by department = 'engineering'
run_filtered_query("What are the policy guidelines and budget rules?", where_filter={"department": "engineering"})

✅ Indexed 4 documents with metadata tags.

🔍 Query: 'What are the policy guidelines and budget rules?' | Filter: None (Unfiltered Search)
   Rank 1 | ID: policy_finance_1 | Dept: finance | Doc Type: company_policy
          Snippet: Company Travel Policy: International travel requests above $2000 require CF...
   Rank 2 | ID: policy_finance_2 | Dept: finance | Doc Type: company_policy
          Snippet: Expense Reimbursement Policy: All expense claims must be filed within 14 da...
   Rank 3 | ID: policy_hr_1 | Dept: hr | Doc Type: company_policy
          Snippet: Remote Work Guidelines: Employees may work remotely up to 2 days per week a...
🔍 Query: 'What are the policy guidelines and budget rules?' | Filter: {'document_type': 'company_policy'}
   Rank 1 | ID: policy_finance_1 | Dept: finance | Doc Type: company_policy
          Snippet: Company Travel Policy: International travel requests above $2000 require CF...
   Rank 2 | ID: policy_finance_2 | Dept: finance | Doc Type: company_p

## 10. Compare Approximate (Chroma HNSW) and Brute-Force (Exact Cosine) Search Behavior

In [71]:
import time
import tracemalloc
import numpy as np

# 1. Prepare benchmark dataset using local reference documents
benchmark_corpus = []

p1 = PROJECT_ROOT / "data" / "profile.txt"
if p1.exists():
    benchmark_corpus.extend([p.strip() for p in p1.read_text(encoding="utf-8").split("\n\n") if p.strip()])

p2 = PROJECT_ROOT / "data" / "profile1.txt"
if p2.exists():
    benchmark_corpus.extend([p.strip() for p in p2.read_text(encoding="utf-8").split("\n\n") if p.strip()])

# Expand dataset to form a benchmark corpus
expanded_corpus = benchmark_corpus * 4
print(f"Total benchmark dataset size: {len(expanded_corpus)} text chunks.")

# 2. Compute embeddings for benchmark dataset
batch_size = 50
corpus_embeddings = []
for i in range(0, len(expanded_corpus), batch_size):
    batch = expanded_corpus[i:i+batch_size]
    res = client.embeddings.create(model=EMBEDDING_MODEL, input=batch)
    corpus_embeddings.extend([item.embedding for item in res.data])

embeddings_matrix = np.array(corpus_embeddings, dtype=np.float32)

# 3. Create Chroma Collection for HNSW Approximate Search
hnsw_collection = client_chroma.get_or_create_collection(name="benchmark_hnsw_vs_bruteforce")
try:
    ex_ids = hnsw_collection.get()["ids"]
    if ex_ids:
        hnsw_collection.delete(ids=ex_ids)
except Exception:
    pass

hnsw_collection.add(
    ids=[f"doc_{idx}" for idx in range(len(expanded_corpus))],
    documents=expanded_corpus,
    embeddings=corpus_embeddings
)

# 4. Perform Benchmark Query
query_str = "Where did John study computer science and what projects did he work on?"
q_emb = np.array(client.embeddings.create(model=EMBEDDING_MODEL, input=query_str).data[0].embedding, dtype=np.float32)
top_k = 5

# --- A. Brute-Force Vector Search (Exact Cosine Similarity) ---
tracemalloc.start()
t0 = time.perf_counter()

norm_matrix = np.linalg.norm(embeddings_matrix, axis=1)
norm_q = np.linalg.norm(q_emb)
cosine_sims = np.dot(embeddings_matrix, q_emb) / (norm_matrix * norm_q)
bf_top_k_indices = np.argsort(cosine_sims)[::-1][:top_k]

t1 = time.perf_counter()
_, bf_peak_mem = tracemalloc.get_traced_memory()
tracemalloc.stop()

bf_latency = (t1 - t0) * 1000
bf_top_ids = [f"doc_{idx}" for idx in bf_top_k_indices]

# --- B. Approximate Vector Search (Chroma HNSW Index) ---
tracemalloc.start()
t2 = time.perf_counter()

hnsw_res = hnsw_collection.query(
    query_embeddings=[q_emb.tolist()],
    n_results=top_k
)

t3 = time.perf_counter()
_, hnsw_peak_mem = tracemalloc.get_traced_memory()
tracemalloc.stop()

hnsw_latency = (t3 - t2) * 1000
hnsw_top_ids = hnsw_res["ids"][0]

# --- C. Compare & Display Performance Metrics ---
top1_match = (bf_top_ids[0] == hnsw_top_ids[0])
overlap_count = len(set(bf_top_ids).intersection(set(hnsw_top_ids)))

print("\n" + "=" * 70)
print("📊 BENCHMARK COMPARISON: BRUTE-FORCE (EXACT) vs APPROXIMATE (HNSW)")
print("=" * 70)
print(f"{'Metric':<25} | {'Brute-Force (Exact)':<20} | {'Chroma (HNSW)':<20}")
print("-" * 70)
print(f"{'Retrieval Latency':<25} | {bf_latency:<20.3f} ms | {hnsw_latency:<20.3f} ms")
print(f"{'Memory Allocated':<25} | {bf_peak_mem / 1024:<20.2f} KB | {hnsw_peak_mem / 1024:<20.2f} KB")
print(f"{'Top 1 ID Match':<25} | {bf_top_ids[0]:<20} | {hnsw_top_ids[0]:<20}")
print("-" * 70)
print(f"First Result Accuracy Match: {'✅ EXACT MATCH' if top1_match else '⚠️ DIFFERENT ORDERING'}")
print(f"Top-{top_k} Neighbor Overlap Rate: {overlap_count}/{top_k} items matched")
print("\nBrute-Force Neighbors: ", bf_top_ids)
print("Chroma HNSW Neighbors:  ", hnsw_top_ids)

Total benchmark dataset size: 596 text chunks.

📊 BENCHMARK COMPARISON: BRUTE-FORCE (EXACT) vs APPROXIMATE (HNSW)
Metric                    | Brute-Force (Exact)  | Chroma (HNSW)       
----------------------------------------------------------------------
Retrieval Latency         | 6.789                ms | 10.773               ms
Memory Allocated          | 3584.26              KB | 161.54               KB
Top 1 ID Match            | doc_345              | doc_280             
----------------------------------------------------------------------
First Result Accuracy Match: ⚠️ DIFFERENT ORDERING
Top-5 Neighbor Overlap Rate: 2/5 items matched

Brute-Force Neighbors:  ['doc_345', 'doc_26', 'doc_196', 'doc_131', 'doc_473']
Chroma HNSW Neighbors:   ['doc_280', 'doc_345', 'doc_408', 'doc_429', 'doc_473']
